In [ ]:
from datetime import datetime
import os 

In [ ]:
data = "/home/jupyter/workspaces/infectiousdiseasephewas2/data/2025-12-13_preparing_cohort_genotype_files_for_regenie_input"

results = "/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"

scratch =  "/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"

#!mkdir {scratch}

In [ ]:
%%bash

#mkdir -p "${scratch}/filtered_pgen"

HWE_and_common_filter_with_plink2 (){

    for chr in {1..22}; do
      plink2 \
        --pfile ${data}/acaf_pgen_files/pgen/acaf_threshold.chr${chr} \
        --threads 8 \
        --maf 0.01 \
        --hwe 1e-6 .001 \
        --geno 0.1 \
        --mind 0.1 \
        --set-missing-var-ids @:# \
        --make-pgen \
        --out ${scratch}/filtered_pgen/acaf_threshold.chr${chr}.qc
        
    done

}

HWE_and_common_filter_with_plink2

In [ ]:
%%bash


count_snps_after_MAF_and_HWE() {
    echo -e "CHR\tLOADED\tAFTER_MAF\tAFTER_HWE(FINAL)"
    echo "------------------------------------------------------------------------"

    # Initialize variables for grand totals
    total_loaded=0
    total_after_maf=0
    total_final=0

    for chr in {1..22}; do
        log_file="${scratch}/filtered_pgen/acaf_threshold.chr${chr}.qc.log"
        
        if [ -f "$log_file" ]; then
            # Extract values
            LOADED=$(grep "variants loaded from" "$log_file" | awk '{print $1}')
            MAF_LOST=$(grep "removed due to allele frequency" "$log_file" | awk '{print $1}')
            HWE_LOST=$(grep "removed due to Hardy-Weinberg" "$log_file" | awk '{print $1}')
            REMAINING=$(grep "variants remaining after main filters" "$log_file" | awk '{print $1}')

            # Calculate step-down
            AFTER_MAF=$((LOADED - MAF_LOST))
            
            # Add to grand totals
            total_loaded=$((total_loaded + LOADED))
            total_after_maf=$((total_after_maf + AFTER_MAF))
            total_final=$((total_final + REMAINING))

            echo -e "chr${chr}\t${LOADED}\t${AFTER_MAF}\t${REMAINING}"
        fi
    done

    echo "------------------------------------------------------------------------"
    echo -e "TOTAL\t${total_loaded}\t${total_after_maf}\t${total_final}"
}


count_snps_after_MAF_and_HWE

In [ ]:
%%bash

scratch="/home/jupyter/workspaces/infectiousdiseasephewas2/scratch/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"

results="/home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input"


prune_and_merge_pgen_files() {

    echo "Pruning PGEN files..."

    # --- STEP 2: LD PRUNE ---
    echo "Starting LD Pruning..."

    # This identifies independent SNPs (r2 < 0.1)
    for chr in {1..22}; do
        plink2 --pfile "${scratch}/filtered_pgen/acaf_threshold.chr${chr}.qc" \
            --indep-pairwise 1000 100 0.1 \
            --threads 8 \
            --out "${scratch}/acaf_threshold.chr${chr}.ld_prune"


        # ---EXTRACT FINAL SET ---
        echo "Extracting final pruned dataset..."


        # merge the pruned SNPs from each chromosome into a final dataset
            plink2 --pfile "${scratch}/filtered_pgen/acaf_threshold.chr${chr}.qc" \
            --threads 8 \
            --extract "${scratch}/acaf_threshold.chr${chr}.ld_prune.prune.in" \
            --make-pgen \
            --out "${scratch}/acaf_threshold.step1_snps_chr${chr}"
    done


    plink2 --pmerge-list <(for chr in {1..22}; do echo "${scratch}/acaf_threshold.step1_snps_chr${chr}"; done) \
        --threads 8 \
        --make-pgen \
        --out "${results}/acaf_threshold.step1_snps"


    # --- STEP 4: FINAL COUNT ---
    FINAL_COUNT2=$(grep -v '^#' "${results}/acaf_threshold.step1_snps.pvar" | wc -l)

    echo "------------------------------------------------------"
    echo "PROCESS COMPLETE"
    echo "Final dataset for REGENIE: ${results}/acaf_threshold.step1_snps"
    echo "Total Independent SNPs: $FINAL_COUNT2"
    echo "------------------------------------------------------"

}


prune_and_merge_pgen_files

In [ ]:
##upload to workspace bucket

In [ ]:
my_bucket = os.getenv('WORKSPACE_BUCKET')
my_bucket

In [ ]:
!gsutil ls "$WORKSPACE_BUCKET/notebooks/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input/"

In [ ]:
!gsutil rm "$WORKSPACE_BUCKET/notebooks/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input/qc_pass.snplist"


In [ ]:
%%bash
# -m = parallel; -r = recursive
gsutil -m cp /home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input/acaf_threshold.step1_snps.log "$WORKSPACE_BUCKET/notebooks/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input/"


In [ ]:
%%bash
# -m = parallel; -r = recursive
gsutil -m cp /home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input/acaf_threshold.step1_snps.psam "$WORKSPACE_BUCKET/notebooks/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input/"


In [ ]:
%%bash
# -m = parallel; -r = recursive
gsutil -m cp /home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input/acaf_threshold.step1_snps.pvar "$WORKSPACE_BUCKET/notebooks/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input/"


In [ ]:
%%bash
# -m = parallel; -r = recursive
gsutil -m cp /home/jupyter/workspaces/infectiousdiseasephewas2/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input/acaf_threshold.step1_snps.pgen "$WORKSPACE_BUCKET/notebooks/results/2025-12-12_preparing_cohort_genotype_files_for_regenie_input/"
